# Impact Split Explainer

This notebook provides a deeper walkthrough of `impact-split` than the README:

1. Problem framing
2. Mathematical engine
3. Worked toy example
4. Tree visualization and segment extraction
5. Parameter sensitivity notes

## 1) Problem Framing

Standard trees are great at separating by average behavior, but for additive KPIs we care about total impact.

Example intuition:

- Segment A: 2 users x -500 average = -1,000 total
- Segment B: 10,000 users x -40 average = -400,000 total

Impact Split is designed to surface Segment B type patterns earlier because business materiality is sum-driven.

## 2) Mathematical Engine

### A. Local Sieve (`delta`)

At each node:

$$
V_{node} = \sum_{j=1}^{N_{node}} |y_j|
$$

$$
\delta = V_{node} \times \text{delta\_pct}
$$

For each category in feature $X_i$:

$$
S_{cat} = \sum_{y \in cat} y
$$

Routing:

- Positive if $S_{cat} > \delta$
- Negative if $S_{cat} < -\delta$
- Neutral otherwise

### B. Split Metric

$$
Gain(X_i) = \frac{|S_P|}{k_P} + \frac{|S_N|}{k_N}
$$

Where $S_P, S_N$ are outer-branch sums and $k_P, k_N$ are the number of categories assigned to each outer branch.

### C. Global Stopping Rule

Let:

$$
V_{global\_P} = \sum_{y_i > 0} y_i,
\quad
V_{global\_N} = \sum_{y_i < 0} |y_i|
$$

At each node:

$$
S_{node\_P} = \sum_{y_i \in node, y_i > 0} y_i,
\quad
S_{node\_N} = \sum_{y_i \in node, y_i < 0} |y_i|
$$

Split only if either ratio is above `min_global_impact_pct`:

- $S_{node\_P} / V_{global\_P}$
- $S_{node\_N} / V_{global\_N}$

In [ ]:
import numpy as np
import pandas as pd

from impact_split import ImpactSplitter

In [ ]:
# Reproducible toy data with additive target signal
rng = np.random.default_rng(42)
n = 5000

regions = np.array(["NCR", "Luzon", "Visayas", "Mindanao"])
channels = np.array(["Direct", "Partner", "Online"])
products = np.array(["A", "B", "C"])

X = pd.DataFrame(
    {
        "region": rng.choice(regions, size=n, p=[0.35, 0.3, 0.2, 0.15]),
        "channel": rng.choice(channels, size=n, p=[0.25, 0.35, 0.4]),
        "product": rng.choice(products, size=n, p=[0.4, 0.35, 0.25]),
    }
)

base = rng.normal(loc=0, scale=35, size=n)

# Add structured impact patterns
impact = (
    np.where((X["region"] == "NCR") & (X["channel"] == "Direct"), 120, 0)
    + np.where((X["region"] == "Mindanao") & (X["product"].isin(["A", "B"])), -95, 0)
    + np.where((X["channel"] == "Online") & (X["product"] == "C"), 35, 0)
)

y = pd.Series(base + impact, name="profit_loss")

X.head(), y.head()

In [ ]:
model = ImpactSplitter(
    delta_pct=0.05,
    min_global_impact_pct=0.01,
    max_depth=4,
)

model.fit(X, y)

In [ ]:
model.plot_tree(figsize=(16, 10))

In [ ]:
segments = model.get_impact_segments()
segments.head(15)

## 3) Parameter Sensitivity Notes

Use these heuristics as a starting point:

- Increase `delta_pct` to make Neutral wider and focus only on very strong category effects.
- Decrease `delta_pct` to make splits more aggressive.
- Increase `min_global_impact_pct` to stop earlier and produce a smaller tree.
- Lower `min_global_impact_pct` only if you need finer-grained segments and can tolerate larger trees.

In [ ]:
# Quick sensitivity run (optional)
for delta_pct, min_global_impact_pct in [(0.03, 0.01), (0.05, 0.01), (0.08, 0.02)]:
    m = ImpactSplitter(
        delta_pct=delta_pct,
        min_global_impact_pct=min_global_impact_pct,
        max_depth=4,
    )
    m.fit(X, y)
    top = m.get_impact_segments().head(3)
    print(f"delta_pct={delta_pct}, min_global_impact_pct={min_global_impact_pct}")
    print(top[["path", "total_sum"]] if {"path", "total_sum"}.issubset(top.columns) else top)
    print("-" * 80)